# KardioSense AI — Notebook 02d: MIMIC-IV ECG External Validation

## Purpose
True external validation of the KardioSense fusion model on a completely
independent real-world dataset — real ECGs + real clinical features + real outcomes.

## Why MIMIC-IV ECG is the gold standard test
```
Dataset PTB-XL (train) CODE-15% (NB02b) MIMIC-IV (this)
─────────────────────────────────────────────────────────────────────────
Country Germany Brazil USA
Hospital Leipzig Univ. HMG Fortaleza Beth Israel Boston
Hardware Schiller Nihon Kohden GE/Philips
Sample rate 500Hz 400Hz 500Hz
ECG labels Cardiologist Algorithm-generated Cardiologist + ICD-10
Clinical data None None Full MIMIC-IV record
N records 21,800 345,779 800,000+
Paired ECG+Clin No No YES ← key
```

## What we test
1. **ECG branch alone**: AUC for MI, AFib, conduction disturbance detection
2. **Clinical branch alone**: 10-year CVD risk from MIMIC demographics + labs
3. **Fusion model**: Combined AUC with real (not synthetic) clinical features
4. **Subgroup analysis**: Performance by age, sex, ethnicity, comorbidity
5. **Attention gate behaviour**: Does α adapt to real clinical data quality?

## Data access — BigQuery + GCS streaming
- ECG metadata + clinical data: BigQuery (`physionet-data.mimiciv_ecg`)
- ECG waveform files: GCS streaming (`gs://physionet-org/files/mimic-iv-ecg/`)
- No downloading required — process in memory, save only embeddings to Drive

## Target sample size: 50,000 ECGs
Provides statistical power for subgroup analysis (500+ positives per subgroup)
while keeping runtime under 2 hours on Colab GPU.

## 0. Setup & Authentication

In [20]:
# ── Authenticate with Google Cloud ────────────────────────────────────────
# Uses the same Google account that has BigQuery access to physionet-data
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

import os
BASE_DIR = os.environ.get('KARDIOSENSE_BASE_DIR', '/content/drive/MyDrive/KardioSenseAI')
CKPT_DIR = f'{BASE_DIR}/checkpoints'
LOG_DIR = f'{BASE_DIR}/logs'
MIMIC_DIR = f'{BASE_DIR}/datasets/mimic_ecg'

for d in [CKPT_DIR, LOG_DIR, MIMIC_DIR]:
    os.makedirs(d, exist_ok=True)

print(' Authenticated and Drive mounted')

# Verify all required checkpoints exist
for path, name in [
    (f'{CKPT_DIR}/kardiosense_ecg_best_mi.pt', 'ECG model (NB02)'),
    (f'{CKPT_DIR}/clinical_model_nhanes_tuned.pkl','Clinical model (NB03b)'),
    (f'{CKPT_DIR}/clinical_shap_nhanes.pkl', 'SHAP explainer (NB03b)'),
    (f'{CKPT_DIR}/nb03b_nhanes_config.json', 'Clinical config'),
    (f'{CKPT_DIR}/fusion_model_best.pt', 'Fusion model (NB04)'),
    (f'{CKPT_DIR}/fusion_config.json', 'Fusion config'),
]:
    exists = os.path.exists(path)
    print(f' {"" if exists else ""} {name}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Authenticated and Drive mounted
  ✅ ECG model (NB02)
  ✅ Clinical model (NB03b)
  ✅ SHAP explainer (NB03b)
  ✅ Clinical config
  ✅ Fusion model (NB04)
  ✅ Fusion config


In [21]:
!pip install -q xgboost shap google-cloud-bigquery wfdb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
import shap, joblib, json, time, warnings, wfdb
warnings.filterwarnings('ignore')

from google.cloud import bigquery
from scipy.signal import butter, filtfilt, resample_poly
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, confusion_matrix,
    brier_score_loss, recall_score, f1_score,
)
from sklearn.calibration import calibration_curve

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : Tesla T4


## 1. Query MIMIC-IV via BigQuery

We pull ECG metadata joined to MIMIC-IV clinical tables:
- `mimiciv_ecg.record_list` — ECG study IDs and file paths
- `mimiciv_hosp.diagnoses_icd` — ICD-10 codes (our CVD outcome)
- `mimiciv_hosp.labevents` — lab results (cholesterol, HbA1c, creatinine)
- `mimiciv_hosp.patients` — demographics (age, sex)
- `mimiciv_hosp.admissions` — ethnicity

**CVD outcome definition (ICD-10):**
- I21/I22: Acute MI
- I20/I25: Chronic IHD / angina
- I50: Heart failure
- I63: Ischaemic stroke
- I48: Atrial fibrillation (for AFib subgroup)

In [22]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

In [23]:
# ── BigQuery project setup ────────────────────────────────────────────────
# Replace with your GCP project ID (visible in BigQuery console top-left)
GCP_PROJECT = os.environ.get('KARDIOSENSE_GCP_PROJECT', 'your-gcp-project-id')  # set to your own PhysioNet-credentialed BigQuery project

client = bigquery.Client(project=GCP_PROJECT)

# ── Query 1: ECG metadata + demographics ─────────────────────────────────
print('Querying ECG metadata + demographics...')

QUERY_METADATA = f"""
SELECT
    ecg.subject_id,
    ecg.study_id,
    ecg.path,
    ecg.ecg_time,
    pat.gender,
    pat.anchor_age AS age,
    adm.race AS ethnicity

FROM `physionet-data.mimiciv_ecg.record_list` AS ecg
JOIN `physionet-data.mimiciv_hosp.patients` AS pat
    ON ecg.subject_id = pat.subject_id
LEFT JOIN (
    SELECT subject_id, race,
           ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY admittime) AS rn
    FROM `physionet-data.mimiciv_hosp.admissions`
) adm ON ecg.subject_id = adm.subject_id AND adm.rn = 1

WHERE pat.anchor_age BETWEEN 30 AND 85
LIMIT 80000
"""

df_meta = client.query(QUERY_METADATA).to_dataframe()
print(f' Metadata: {len(df_meta):,} ECG records')
print(f' Age range: {df_meta["age"].min():.0f}–{df_meta["age"].max():.0f} yrs')
print(f' Sex: {dict(df_meta["gender"].value_counts())}')
print(f' Ethnicities: {df_meta["ethnicity"].value_counts().head(5).to_dict()}')

Querying ECG metadata + demographics...


Forbidden: 403 Access Denied: Table physionet-data:mimiciv_hosp.admissions: User does not have permission to query table physionet-data:mimiciv_hosp.admissions, or perhaps it does not exist.; reason: accessDenied, message: Access Denied: Table physionet-data:mimiciv_hosp.admissions: User does not have permission to query table physionet-data:mimiciv_hosp.admissions, or perhaps it does not exist.

Location: US
Job ID: 87fe839a-6199-4060-9410-34124652ed2e


In [ ]:
# ── Query 2: CVD outcomes from ICD-10 codes ──────────────────────────────
print('Querying CVD outcomes from ICD-10 diagnoses...')

QUERY_CVD = f"""
SELECT DISTINCT
    subject_id,
    MAX(CASE
        WHEN REGEXP_CONTAINS(icd_code, r'^I2[012]') THEN 1 -- Acute MI
        WHEN REGEXP_CONTAINS(icd_code, r'^I2[05]') THEN 1 -- Chronic IHD
        WHEN REGEXP_CONTAINS(icd_code, r'^I50') THEN 1 -- Heart failure
        WHEN REGEXP_CONTAINS(icd_code, r'^I63') THEN 1 -- Stroke
        ELSE 0 END) AS cvd_event,
    MAX(CASE
        WHEN REGEXP_CONTAINS(icd_code, r'^I2[012]') THEN 1
        ELSE 0 END) AS has_mi,
    MAX(CASE
        WHEN REGEXP_CONTAINS(icd_code, r'^I48') THEN 1
        ELSE 0 END) AS has_afib,
    MAX(CASE
        WHEN REGEXP_CONTAINS(icd_code, r'^I[34][45]') THEN 1 -- BBB
        WHEN REGEXP_CONTAINS(icd_code, r'^I44') THEN 1 -- AV block
        ELSE 0 END) AS has_cd

FROM `physionet-data.mimiciv_hosp.diagnoses_icd`
WHERE icd_version = 10
GROUP BY subject_id
"""

df_cvd = client.query(QUERY_CVD).to_dataframe()
print(f' CVD outcomes: {len(df_cvd):,} patients')
print(f' CVD positive : {df_cvd["cvd_event"].sum():,} ({df_cvd["cvd_event"].mean():.1%})')
print(f' MI : {df_cvd["has_mi"].sum():,}')
print(f' AFib : {df_cvd["has_afib"].sum():,}')
print(f' Conduction : {df_cvd["has_cd"].sum():,}')

In [ ]:
# ── Query 3: Lab values (cholesterol, HbA1c, creatinine) ─────────────────
print('Querying lab values...')

# LOINC codes:
# 2093-3 = Total Cholesterol
# 2085-9 = HDL
# 13457-7 = LDL
# 4548-4 = HbA1c
# 2160-0 = Creatinine (for eGFR)
QUERY_LABS = f"""
SELECT
    subject_id,
    MAX(CASE WHEN itemid IN (50912) THEN valuenum END) AS creatinine,
    MAX(CASE WHEN itemid IN (50813) THEN valuenum END) AS lactate,
    AVG(CASE WHEN itemid IN (50907) THEN valuenum END) AS total_cholesterol_mgdl,
    AVG(CASE WHEN itemid IN (50904) THEN valuenum END) AS hdl_mgdl,
    AVG(CASE WHEN itemid IN (50906) THEN valuenum END) AS ldl_mgdl,
    AVG(CASE WHEN itemid IN (50852) THEN valuenum END) AS hba1c

FROM `physionet-data.mimiciv_hosp.labevents`
WHERE itemid IN (50907, 50904, 50906, 50852, 50912)
  AND valuenum IS NOT NULL
  AND valuenum > 0
GROUP BY subject_id
"""

df_labs = client.query(QUERY_LABS).to_dataframe()
# Convert mg/dL → mmol/L
df_labs['total_cholesterol'] = df_labs['total_cholesterol_mgdl'] / 38.67
df_labs['hdl_cholesterol'] = df_labs['hdl_mgdl'] / 38.67
df_labs['ldl_cholesterol'] = df_labs['ldl_mgdl'] / 38.67

print(f' Labs: {len(df_labs):,} patients with at least one lab value')
miss = df_labs[['total_cholesterol','hdl_cholesterol','ldl_cholesterol','hba1c']].isna().mean()*100
for feat, pct in miss.items():
    print(f' {feat:25s}: {pct:.1f}% missing')

In [ ]:
# ── Query 4: BP medications ───────────────────────────────────────────────
print('Querying BP medications...')

QUERY_MEDS = f"""
SELECT DISTINCT subject_id,
    1 AS on_bp_medication
FROM `physionet-data.mimiciv_hosp.prescriptions`
WHERE LOWER(drug) LIKE '%amlodipine%'
   OR LOWER(drug) LIKE '%lisinopril%'
   OR LOWER(drug) LIKE '%metoprolol%'
   OR LOWER(drug) LIKE '%atenolol%'
   OR LOWER(drug) LIKE '%losartan%'
   OR LOWER(drug) LIKE '%hydrochlorothiazide%'
   OR LOWER(drug) LIKE '%ramipril%'
   OR LOWER(drug) LIKE '%bisoprolol%'
"""

df_meds = client.query(QUERY_MEDS).to_dataframe()
print(f' BP medication records: {len(df_meds):,} patients')

In [ ]:
# ── Query 5: Blood pressure from chartevents ─────────────────────────────
print('Querying blood pressure readings...')

QUERY_BP = f"""
SELECT
    subject_id,
    AVG(CASE WHEN itemid IN (220179, 224167) THEN valuenum END) AS systolic_bp,
    AVG(CASE WHEN itemid IN (220180, 224643) THEN valuenum END) AS diastolic_bp

FROM `physionet-data.mimiciv_icu.chartevents`
WHERE itemid IN (220179, 224167, 220180, 224643)
  AND valuenum BETWEEN 50 AND 300
GROUP BY subject_id
"""

try:
    df_bp = client.query(QUERY_BP).to_dataframe()
    print(f' BP readings: {len(df_bp):,} patients')
    print(f' Mean SBP: {df_bp["systolic_bp"].mean():.1f} mmHg')
except Exception as e:
    # chartevents is very large — may timeout, use fallback
    print(f' chartevents query timed out: {e}')
    print('Using admission vitals as fallback...')
    df_bp = pd.DataFrame(columns=['subject_id','systolic_bp','diastolic_bp'])

In [ ]:
# ── Merge all tables ─────────────────────────────────────────────────────
print('\nMerging all tables...')

df = df_meta.copy()
df = df.merge(df_cvd, on='subject_id', how='left')
df = df.merge(df_labs, on='subject_id', how='left')
df = df.merge(df_meds, on='subject_id', how='left')
if len(df_bp) > 0:
    df = df.merge(df_bp, on='subject_id', how='left')
else:
    df['systolic_bp'] = np.nan
    df['diastolic_bp'] = np.nan

# Fill missing outcomes as negative
df['cvd_event'] = df['cvd_event'].fillna(0).astype(int)
df['has_mi'] = df['has_mi'].fillna(0).astype(int)
df['has_afib'] = df['has_afib'].fillna(0).astype(int)
df['has_cd'] = df['has_cd'].fillna(0).astype(int)
df['on_bp_medication'] = df['on_bp_medication'].fillna(0).astype(int)

# Sex: M=1, F=0
df['sex'] = (df['gender'] == 'M').astype(float)

# Ethnicity flags
eth = df['ethnicity'].fillna('').str.upper()
df['black_african'] = eth.str.contains('BLACK|AFRICAN').astype(float)
df['hispanic'] = eth.str.contains('HISPANIC|LATINO').astype(float)

# Impute BP if missing
df['systolic_bp'] = df['systolic_bp'].fillna(130.0)
df['diastolic_bp'] = df['diastolic_bp'].fillna(80.0)

# Fill lipids with population medians
df['total_cholesterol'] = df['total_cholesterol'].fillna(df['total_cholesterol'].median())
df['hdl_cholesterol'] = df['hdl_cholesterol'].fillna(df['hdl_cholesterol'].median())
df['ldl_cholesterol'] = df['ldl_cholesterol'].fillna(df['ldl_cholesterol'].median())
df['hba1c'] = df['hba1c'].fillna(df['hba1c'].median())

print(f' Merged dataset: {len(df):,} rows')
print(f' CVD positive : {df["cvd_event"].sum():,} ({df["cvd_event"].mean():.1%})')
print(f' MI positive : {df["has_mi"].sum():,}')
print(f' AFib positive : {df["has_afib"].sum():,}')

# Sample up to 50,000 — stratified on CVD outcome
N_SAMPLE = 50_000
if len(df) > N_SAMPLE:
    pos = df[df.cvd_event == 1].sample(min(df.cvd_event.sum(), N_SAMPLE//4), random_state=42)
    neg = df[df.cvd_event == 0].sample(N_SAMPLE - len(pos), random_state=42)
    df_val = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)
else:
    df_val = df.copy()

print(f'\nValidation sample: {len(df_val):,} '
      f'(CVD rate: {df_val["cvd_event"].mean():.1%})')

# Save metadata to Drive
df_val.to_csv(f'{MIMIC_DIR}/mimic_validation_metadata.csv', index=False)
print(f'Metadata saved to Drive.')

## 2. Rebuild Models & Load Checkpoints

In [ ]:
# ── ECG Model ────────────────────────────────────────────────────────────
class ResBlock1D(nn.Module):
    def __init__(self,in_ch,out_ch,kernel_size=7,stride=1,dropout=0.2):
        super().__init__()
        pad=kernel_size//2
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel_size,stride=stride,padding=pad,bias=False)
        self.bn1=nn.BatchNorm1d(out_ch)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel_size,stride=1,padding=pad,bias=False)
        self.bn2=nn.BatchNorm1d(out_ch)
        self.drop=nn.Dropout(dropout)
        self.skip=nn.Sequential(
            nn.Conv1d(in_ch,out_ch,1,stride=stride,bias=False),nn.BatchNorm1d(out_ch)
        ) if (in_ch!=out_ch or stride!=1) else nn.Identity()
    def forward(self,x):
        r=self.skip(x); o=F.relu(self.bn1(self.conv1(x))); o=self.drop(o)
        return F.relu(self.bn2(self.conv2(o))+r)

class AttentionPool(nn.Module):
    def __init__(self,d): super().__init__(); self.a=nn.Linear(d,1)
    def forward(self,x):
        w=torch.softmax(self.a(x),dim=1); return (x*w).sum(dim=1),w.squeeze(-1)

class KardioSenseECGModel(nn.Module):
    def __init__(self,n_leads=6,n_classes=5,dropout=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(n_leads,32,15,stride=2,padding=7,bias=False),
            nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(3,stride=2,padding=1))
        self.layer1=ResBlock1D(32,64,stride=2,dropout=dropout)
        self.layer2=ResBlock1D(64,128,stride=2,dropout=dropout)
        self.layer3=ResBlock1D(128,256,stride=2,dropout=dropout)
        self.layer4=ResBlock1D(256,256,stride=2,dropout=dropout)
        self.bilstm=nn.LSTM(256,128,num_layers=2,batch_first=True,
                             bidirectional=True,dropout=dropout)
        self.self_attn=nn.MultiheadAttention(256,4,dropout=dropout,batch_first=True)
        self.attn_norm=nn.LayerNorm(256)
        self.attn_pool=AttentionPool(256)
        self.mask_proj=nn.Sequential(nn.Linear(n_leads,32),nn.ReLU())
        self.classifier=nn.Sequential(nn.Linear(288,128),nn.ReLU(),
                                       nn.Dropout(dropout),nn.Linear(128,n_classes))
    def _encode(self,sig,mask):
        x=self.layer4(self.layer3(self.layer2(self.layer1(self.stem(sig)))))
        x=x.permute(0,2,1); x,_=self.bilstm(x)
        ao,aw=self.self_attn(x,x,x); x=self.attn_norm(x+ao)
        emb,pw=self.attn_pool(x); return emb,pw,aw
    def forward(self,sig,mask):
        emb,pw,_=self._encode(sig,mask)
        return self.classifier(torch.cat([emb,self.mask_proj(mask)],dim=-1)),pw
    def get_ecg_embedding(self,sig,mask):
        with torch.no_grad(): emb,_,_=self._encode(sig,mask)
        return emb

ecg_ckpt = torch.load(f'{CKPT_DIR}/kardiosense_ecg_best_mi.pt',
                         map_location=DEVICE,weights_only=True)
ecg_cfg = ecg_ckpt['config']
ecg_model = KardioSenseECGModel(n_leads=ecg_cfg['n_leads'],
                                  n_classes=ecg_cfg['n_classes']).to(DEVICE)
ecg_model.load_state_dict(ecg_ckpt['model_state'])
ecg_model.eval()
SIX_LEAD_INDICES = ecg_cfg['lead_indices']
LABEL_NAMES = ecg_cfg['label_names']
print(f' ECG model epoch={ecg_ckpt["epoch"]} MI AUC={ecg_ckpt["mi_auc"]:.4f}')

# ── Clinical + SHAP ────────────────────────────────────────────────────────
clinical_model = joblib.load(f'{CKPT_DIR}/clinical_model_nhanes_tuned.pkl')
shap_explainer = joblib.load(f'{CKPT_DIR}/clinical_shap_nhanes.pkl')
with open(f'{CKPT_DIR}/nb03b_nhanes_config.json') as f:
    clin_cfg = json.load(f)
CLINICAL_FEATURE_NAMES = clin_cfg['feature_names']
print(f' Clinical model AUC={clin_cfg["test_auc"]:.4f}')

In [ ]:
# ── Fusion Model ─────────────────────────────────────────────────────────
with open(f'{CKPT_DIR}/fusion_config.json') as f:
    fusion_cfg = json.load(f)

ECG_DIM = fusion_cfg['ecg_dim'] # 256
CLIN_DIM = fusion_cfg['clin_dim'] # 32
FUSE_DIM = fusion_cfg['fusion_dim'] # 128

class AttentionGate(nn.Module):
    def __init__(self,ecg_dim=256,clin_dim=32,quality_dim=2,fusion_dim=128):
        super().__init__()
        self.ecg_proj = nn.Sequential(nn.Linear(ecg_dim,fusion_dim),nn.LayerNorm(fusion_dim),nn.GELU())
        self.clin_proj = nn.Sequential(nn.Linear(clin_dim,fusion_dim),nn.LayerNorm(fusion_dim),nn.GELU())
        self.gate_net = nn.Sequential(nn.Linear(fusion_dim*2+quality_dim,64),nn.ReLU(),
                                        nn.Dropout(0.2),nn.Linear(64,2))
        self.cross_attn = nn.MultiheadAttention(embed_dim=fusion_dim,num_heads=4,
                                                  dropout=0.1,batch_first=True)
    def forward(self,ecg_emb,clin_emb,quality):
        e=self.ecg_proj(ecg_emb); c=self.clin_proj(clin_emb)
        ec=torch.stack([e,c],dim=1); ec,_=self.cross_attn(ec,ec,ec)
        e2,c2=ec[:,0,:],ec[:,1,:]
        gates=torch.sigmoid(self.gate_net(torch.cat([e2,c2,quality],dim=-1)))
        alpha,beta=gates[:,0:1],gates[:,1:2]
        denom=alpha+beta+1e-8
        fused=alpha/denom*e2 + beta/denom*c2
        return fused,alpha.squeeze(1),beta.squeeze(1)

class KardioSenseFusionModel(nn.Module):
    def __init__(self,ecg_dim=256,clin_dim=32,fusion_dim=128,dropout=0.3):
        super().__init__()
        self.gate = AttentionGate(ecg_dim,clin_dim,2,fusion_dim)
        self.classifier = nn.Sequential(nn.Linear(fusion_dim,64),nn.GELU(),
            nn.Dropout(dropout),nn.Linear(64,32),nn.GELU(),
            nn.Dropout(dropout),nn.Linear(32,1))
    def forward(self,ecg_emb,clin_emb,quality):
        fused,alpha,beta=self.gate(ecg_emb,clin_emb,quality)
        return self.classifier(fused),alpha,beta

fusion_model = KardioSenseFusionModel(ECG_DIM,CLIN_DIM,FUSE_DIM).to(DEVICE)
fuse_ckpt = torch.load(f'{CKPT_DIR}/fusion_model_best.pt',
                            map_location=DEVICE,weights_only=True)
fusion_model.load_state_dict(fuse_ckpt['model_state'])
fusion_model.eval()
print(f' Fusion model epoch={fuse_ckpt["epoch"]} val_auc={fuse_ckpt["val_auc"]:.4f}')

## 3. Signal Preprocessing Pipeline

MIMIC-IV ECG signals are stored as WFDB files on GCS.
We stream each file, apply the same preprocessing as training
(bandpass 0.5–40Hz, z-score per lead, resample to 500Hz if needed),
and extract the 256-dim ECG embedding immediately to save memory.

In [ ]:
import io, requests, struct

# MIMIC ECG GCS base path
MIMIC_GCS_BASE = 'https://physionet.org/files/mimic-iv-ecg/1.0'

# Bandpass filter — same as training
def make_bandpass(lowcut=0.5, highcut=40.0, fs=500, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return b, a

BP_B, BP_A = make_bandpass()


def load_mimic_ecg(path: str, max_retries: int = 3) -> np.ndarray | None:
    """
    Stream a MIMIC-IV ECG record from PhysioNet GCS.

    path: relative path from record_list, e.g. 'files/p10/p10000032/s10000032'
    Returns: (5000, 12) float32 preprocessed signal, or None on failure
    """
    # Build header URL
    base_url = f'{MIMIC_GCS_BASE}/{path}'
    hea_url = f'{base_url}.hea'
    dat_url = f'{base_url}.dat'

    for attempt in range(max_retries):
        try:
            # Download header to get signal info
            hea_resp = requests.get(hea_url, timeout=15)
            if hea_resp.status_code != 200:
                return None

            # Parse header
            hea_lines = hea_resp.text.strip().split('\n')
            header_info = hea_lines[0].split()
            n_leads = int(header_info[1])
            fs_orig = int(header_info[2])
            n_samples = int(header_info[3]) if len(header_info) > 3 else fs_orig * 10

            # Parse gain and baseline per lead
            gains, baselines = [], []
            for line in hea_lines[1:n_leads+1]:
                parts = line.split()
                if len(parts) >= 3:
                    gain_str = parts[2].split('/')[0].replace('(0)','')
                    gains.append(float(gain_str) if gain_str else 200.0)
                    baselines.append(int(parts[4]) if len(parts) > 4 else 0)
                else:
                    gains.append(200.0); baselines.append(0)

            # Lead name mapping — MIMIC uses standard names
            lead_names = []
            for line in hea_lines[1:n_leads+1]:
                parts = line.split()
                lead_names.append(parts[-1] if len(parts) >= 1 else f'L{len(lead_names)}')

            # Download binary signal data
            dat_resp = requests.get(dat_url, timeout=30)
            if dat_resp.status_code != 200:
                return None

            raw_bytes = dat_resp.content
            # WFDB format 16: 2 bytes per sample, little-endian signed int
            total_samples = n_samples * n_leads
            if len(raw_bytes) < total_samples * 2:
                return None

            raw_int = np.frombuffer(raw_bytes[:total_samples*2], dtype=np.int16)
            signal = raw_int.reshape(n_samples, n_leads).astype(np.float32)

            # Apply gain correction → millivolts → normalise
            for i in range(min(n_leads, len(gains))):
                signal[:, i] = (signal[:, i] - baselines[i]) / gains[i]

            # Map MIMIC lead order to PTB-XL order
            MIMIC_TO_PTBXL = {
                'I':0,'II':1,'III':2,'aVR':3,'aVL':4,'aVF':5,
                'V1':6,'V2':7,'V3':8,'V4':9,'V5':10,'V6':11,
            }
            signal_12 = np.zeros((n_samples, 12), dtype=np.float32)
            for i, lname in enumerate(lead_names):
                lname_clean = lname.strip().upper().replace('AVR','aVR').replace('AVL','aVL').replace('AVF','aVF')
                if lname_clean in MIMIC_TO_PTBXL:
                    signal_12[:, MIMIC_TO_PTBXL[lname_clean]] = signal[:, i]

            # Resample to 500Hz if needed
            if fs_orig != 500:
                from math import gcd
                g = gcd(500, fs_orig)
                up = 500 // g; dn = fs_orig // g
                resampled = np.zeros((5000, 12), dtype=np.float32)
                for lead in range(12):
                    r = resample_poly(signal_12[:, lead].astype(np.float64), up, dn)
                    resampled[:, lead] = r[:5000] if len(r) >= 5000 else np.pad(r, (0, 5000-len(r)))
                signal_12 = resampled
            else:
                # Crop or pad to exactly 5000 samples
                if n_samples >= 5000:
                    signal_12 = signal_12[:5000]
                else:
                    pad = np.zeros((5000-n_samples, 12), dtype=np.float32)
                    signal_12 = np.vstack([signal_12, pad])

            # Apply bandpass filter + z-score normalise per lead
            for lead in range(12):
                x = signal_12[:, lead].astype(np.float64)
                if x.std() > 1e-8:
                    x = filtfilt(BP_B, BP_A, x)
                    x = (x - x.mean()) / (x.std() + 1e-8)
                signal_12[:, lead] = x.astype(np.float32)

            return signal_12 # (5000, 12) preprocessed

        except Exception as e:
            if attempt == max_retries - 1:
                return None
            time.sleep(1)
    return None


# ── Test on first record ────────────────────────────────────────────────────
print('Testing signal loader on first record...')
test_path = df_val['path'].iloc[0]
test_sig = load_mimic_ecg(test_path)
if test_sig is not None:
    print(f' Signal loaded: {test_sig.shape} '
          f'mean={test_sig.mean():.4f} std={test_sig.std():.4f}')
    # Check for flat signals
    energy = np.mean(test_sig**2)
    print(f' Signal energy: {energy:.4f} (>0.05 = usable)')
else:
    print(' Signal load failed — check PhysioNet access and path format')
    print(f' Path attempted: {test_path}')

## 4. Build Clinical Feature Matrix

In [ ]:
def build_mimic_clinical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build clinical feature matrix from MIMIC merged data."""
    out = pd.DataFrame(index=df.index)

    out['age'] = pd.to_numeric(df['age'], errors='coerce').fillna(55)
    out['sex'] = df['sex'].fillna(0.5)
    out['systolic_bp'] = df['systolic_bp'].fillna(130)
    out['diastolic_bp'] = df['diastolic_bp'].fillna(80)
    out['bmi'] = df.get('bmi', pd.Series(np.full(len(df), 27.5))).fillna(27.5)
    out['smoking_current'] = 0.0
    out['smoking_ever'] = 0.0
    out['diabetes'] = (df.get('hba1c', pd.Series(np.zeros(len(df)))).fillna(5.7) >= 6.5).astype(float)
    out['prediabetes'] = ((df.get('hba1c', pd.Series(np.zeros(len(df)))).fillna(5.7) >= 5.7) &
                              (df.get('hba1c', pd.Series(np.zeros(len(df)))).fillna(5.7) < 6.5)).astype(float)
    out['on_bp_medication'] = df['on_bp_medication'].fillna(0)
    out['total_cholesterol'] = df['total_cholesterol'].fillna(5.2)
    out['hdl_cholesterol'] = df['hdl_cholesterol'].fillna(1.3)
    out['ldl_cholesterol'] = df['ldl_cholesterol'].fillna(3.0)
    out['non_hdl_chol'] = out['total_cholesterol'] - out['hdl_cholesterol']
    out['chol_hdl_ratio'] = out['total_cholesterol'] / (out['hdl_cholesterol'] + 0.1)
    out['hba1c'] = df.get('hba1c', pd.Series(np.full(len(df), 5.7))).fillna(5.7)

    out['pulse_pressure'] = out['systolic_bp'] - out['diastolic_bp']
    out['obese'] = (out['bmi'] >= 30).astype(float)
    out['age_over_55'] = (out['age'] >= 55).astype(float)
    out['age_over_65'] = (out['age'] >= 65).astype(float)
    out['hypertension_stage2'] = ((out['systolic_bp'] >= 160) | (out['diastolic_bp'] >= 100)).astype(float)
    out['africa_bp_score'] = ((out['systolic_bp'] - 120) * 0.022).clip(lower=0)
    out['africa_age_score'] = np.where(out['age'] < 45, out['age']*0.12, out['age']*0.08)
    out['black_african'] = df['black_african'].fillna(0)
    out['hispanic'] = df['hispanic'].fillna(0)
    out['age_x_sbp'] = out['age'] * out['systolic_bp']
    out['diabetes_x_sbp'] = out['diabetes'] * out['systolic_bp']
    out['hdl_x_age'] = out['hdl_cholesterol'] * out['age']
    out['bmi_x_sbp'] = out['bmi'] * out['systolic_bp']
    out['smoke_x_chol'] = out['smoking_current'] * out['total_cholesterol']
    out['hba1c_x_bmi'] = out['hba1c'] * out['bmi']
    out['dbp_x_sbp'] = out['diastolic_bp'] * out['systolic_bp']
    out['bmi_x_age'] = out['bmi'] * out['age']

    return out.reindex(columns=CLINICAL_FEATURE_NAMES, fill_value=0.0)


clin_features_mimic = build_mimic_clinical_features(df_val)
print(f' Clinical features: {clin_features_mimic.shape}')

# Clinical completeness — what's actually real vs imputed
real_feats = ['age','sex','systolic_bp','diastolic_bp','total_cholesterol',
              'hdl_cholesterol','hba1c','on_bp_medication']
print('\nFeature source (real MIMIC data vs imputed):')
for feat in real_feats:
    src = df_val.get(feat, pd.Series([np.nan]*len(df_val)))
    real_pct = src.notna().mean() * 100
    print(f' {feat:25s}: {real_pct:.1f}% real MIMIC data')

## 5. Run Full Validation — ECG + Clinical + Fusion

This is the main loop. For each record:
1. Stream ECG signal from PhysioNet GCS
2. Extract 256-dim ECG embedding
3. Compute 32-dim clinical SHAP embedding
4. Run fusion model → CVD probability
5. Record all predictions

**Runtime estimate:** ~2–3 hours for 50,000 records on Colab GPU.
Results are checkpointed every 500 records to Drive.
If Colab disconnects, restart from checkpoint.

In [ ]:
RESULTS_PATH = f'{MIMIC_DIR}/mimic_validation_results.npz'
BATCH_SIZE = 32 # ECG streaming: process in small batches

# Check for existing checkpoint
if os.path.exists(RESULTS_PATH):
    print(' Results checkpoint found — loading...')
    res = np.load(RESULTS_PATH, allow_pickle=True)
    all_ecg_preds = list(res['ecg_preds'])
    all_clin_preds = list(res['clin_preds'])
    all_fusion_preds = list(res['fusion_preds'])
    all_labels = list(res['labels'])
    all_mi_labels = list(res['mi_labels'])
    all_afib_labels = list(res['afib_labels'])
    all_alphas = list(res['alphas'])
    all_betas = list(res['betas'])
    start_idx = int(res['n_processed'])
    print(f' Resuming from record {start_idx:,}')
else:
    all_ecg_preds, all_clin_preds, all_fusion_preds = [], [], []
    all_labels, all_mi_labels, all_afib_labels = [], [], []
    all_alphas, all_betas = [], []
    start_idx = 0

n_errors = 0
n_flat = 0
t0 = time.time()

print(f'\n Running validation on {len(df_val):,} MIMIC-IV ECG records...')
print(f' Starting from record {start_idx:,}')
print(f' Checkpointing every 500 records to Drive')
print()

for i in range(start_idx, len(df_val)):
    row = df_val.iloc[i]
    path = row['path']

    # ── 1. Stream ECG signal ─────────────────────────────────────────
    sig_np = load_mimic_ecg(path)

    if sig_np is None:
        n_errors += 1
        continue

    # Check signal quality — skip flat signals
    energy = np.mean(sig_np**2)
    if energy < 0.02:
        n_flat += 1
        continue

    # ── 2. ECG embedding ─────────────────────────────────────────────
    sig_t = torch.tensor(
        sig_np[:, SIX_LEAD_INDICES].T[np.newaxis], # (1,6,5000)
        dtype=torch.float32
    ).to(DEVICE)
    msk_t = torch.ones(1, len(SIX_LEAD_INDICES), dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        ecg_emb = ecg_model.get_ecg_embedding(sig_t, msk_t) # (1,256)
        ecg_logits, _ = ecg_model(sig_t, msk_t)
        ecg_probs = torch.sigmoid(ecg_logits).cpu().numpy()[0]

    # ── 3. Clinical embedding ─────────────────────────────────────────
    clin_row = clin_features_mimic.iloc[i:i+1]
    clin_emb = shap_explainer.shap_values(
        clin_row.reindex(columns=CLINICAL_FEATURE_NAMES, fill_value=0).values
    )
    # Top 32 by global importance
    global_imp = np.abs(clin_emb).mean(axis=0)
    top32 = np.argsort(global_imp)[::-1][:32]
    clin_emb32 = clin_emb[:, top32].astype(np.float32)

    clin_prob = float(clinical_model.predict_proba(
        clin_row.reindex(columns=CLINICAL_FEATURE_NAMES, fill_value=0).values
    )[0, 1])

    # ── 4. Fusion ─────────────────────────────────────────────────────
    # Clinical quality: fraction of non-imputed features
    n_real = sum(1 for f in ['total_cholesterol','hdl_cholesterol',
                               'hba1c','on_bp_medication']
                  if pd.notna(row.get(f)))
    clin_quality = n_real / 4.0

    clin_emb_t = torch.tensor(clin_emb32, dtype=torch.float32).to(DEVICE)
    qual_t = torch.tensor([[1.0, clin_quality]], dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        logit, alpha, beta = fusion_model(ecg_emb, clin_emb_t, qual_t)
        fusion_prob = float(torch.sigmoid(logit).item())

    # ── Store results ─────────────────────────────────────────────────
    all_ecg_preds.append(float(ecg_probs[LABEL_NAMES.index('MI')])) # MI score
    all_clin_preds.append(clin_prob)
    all_fusion_preds.append(fusion_prob)
    all_labels.append(int(row['cvd_event']))
    all_mi_labels.append(int(row.get('has_mi', 0)))
    all_afib_labels.append(int(row.get('has_afib', 0)))
    all_alphas.append(float(alpha.item()))
    all_betas.append(float(beta.item()))

    # ── Progress + checkpoint ─────────────────────────────────────────
    n_done = len(all_fusion_preds)
    if n_done % 500 == 0:
        elapsed = time.time() - t0
        eta = elapsed / n_done * (len(df_val) - i - 1)
        print(f' [{i+1:,}/{len(df_val):,}] '
              f'done={n_done:,} errors={n_errors} flat={n_flat} '
              f'elapsed={elapsed/60:.1f}min ETA={eta/60:.0f}min')
        # Checkpoint
        np.savez_compressed(RESULTS_PATH,
            ecg_preds = np.array(all_ecg_preds),
            clin_preds = np.array(all_clin_preds),
            fusion_preds = np.array(all_fusion_preds),
            labels = np.array(all_labels),
            mi_labels = np.array(all_mi_labels),
            afib_labels = np.array(all_afib_labels),
            alphas = np.array(all_alphas),
            betas = np.array(all_betas),
            n_processed = np.array(i+1))

# Final arrays
ecg_preds = np.array(all_ecg_preds, dtype=np.float32)
clin_preds = np.array(all_clin_preds, dtype=np.float32)
fusion_preds = np.array(all_fusion_preds, dtype=np.float32)
labels = np.array(all_labels, dtype=np.int32)
mi_labels = np.array(all_mi_labels, dtype=np.int32)
afib_labels = np.array(all_afib_labels, dtype=np.int32)
alphas = np.array(all_alphas, dtype=np.float32)
betas = np.array(all_betas, dtype=np.float32)

print(f'\n Validation complete: {len(fusion_preds):,} records processed')
print(f' Errors (load failed): {n_errors}')
print(f' Flat signals skipped: {n_flat}')
print(f' CVD positive rate : {labels.mean():.3f}')

## 6. Full External Validation Metrics

In [ ]:
# ── Threshold ────────────────────────────────────────────────────────────
prec_f, rec_f, thresh_f = precision_recall_curve(labels, fusion_preds)
r70_idx = np.argmin(np.abs(rec_f - 0.70))
THRESH = float(thresh_f[min(r70_idx, len(thresh_f)-1)])
y_pred = (fusion_preds >= THRESH).astype(int)
cm = confusion_matrix(labels, y_pred)
tn,fp,fn,tp = cm.ravel() if cm.size==4 else (0,0,0,0)

fusion_auc = roc_auc_score(labels, fusion_preds)
ecg_auc = roc_auc_score(labels, ecg_preds) if labels.sum()>0 else np.nan
clin_auc = roc_auc_score(labels, clin_preds) if labels.sum()>0 else np.nan
fusion_ap = average_precision_score(labels, fusion_preds)
fusion_brier = brier_score_loss(labels, fusion_preds)
fusion_rec = recall_score(labels, y_pred)
fusion_prec = float(tp/(tp+fp+1e-8))
fusion_f1 = f1_score(labels, y_pred)

print('='*76)
print('MIMIC-IV EXTERNAL VALIDATION — REAL PAIRED ECG + CLINICAL DATA')
print('='*76)
print(f' n={len(labels):,} CVD rate={labels.mean():.3f} Threshold={THRESH:.3f}')
print()
metrics_table = [
    ('AUC (Fusion)', fusion_auc, 0.85, 'Target ≥0.85'),
    ('AUC (ECG only)', ecg_auc, None, 'ECG branch standalone'),
    ('AUC (Clinical)', clin_auc, None, 'Clinical branch standalone'),
    ('AP', fusion_ap, None, 'Area under PR curve'),
    ('Brier', fusion_brier,None, 'Calibration error'),
    ('Recall', fusion_rec, 0.70, 'CVD patients caught'),
    ('Precision', fusion_prec, None, 'Flagged correctly'),
    ('F1', fusion_f1, None, 'Harmonic mean'),
]
for name,val,target,note in metrics_table:
    flag = ''
    if target is not None:
        flag = ' ' if val>=target else ' '
    print(f' {name:20s}: {val:.4f}{flag:4s} {note}')
print(f' TP={tp} FP={fp} FN={fn} TN={tn}')
print()
print('Performance vs training datasets:')
print(f' PTB-XL (train) AUC = {ecg_ckpt["mi_auc"]:.4f} [ECG branch]')
print(f' NHANES (train) AUC = {clin_cfg["test_auc"]:.4f} [Clinical branch]')
print(f' CODE-15% (NB02b) AUC = see NB02b [ECG external, no clinical]')
print(f' MIMIC-IV (this) AUC = {fusion_auc:.4f} [FULL FUSION, real data]')
print('='*76)

## 7. Subgroup Analysis
This is what clinical papers require — performance must be equitable across groups.

In [ ]:
# Subgroup definitions from MIMIC metadata
df_results = df_val.iloc[:len(labels)].copy()
df_results = df_results[df_results.index.isin(range(len(labels)))].reset_index(drop=True)
df_results['fusion_pred'] = fusion_preds
df_results['ecg_pred'] = ecg_preds
df_results['clin_pred'] = clin_preds
df_results['label'] = labels
df_results['alpha'] = alphas
df_results['beta'] = betas
df_results['age_group'] = pd.cut(df_results['age'],
                                     bins=[30,45,55,65,75,85],
                                     labels=['30–45','45–55','55–65','65–75','75–85'])

subgroups = [
    ('All patients', df_results),
    ('Male', df_results[df_results.sex==1]),
    ('Female', df_results[df_results.sex==0]),
    ('Age 30–55', df_results[df_results.age<55]),
    ('Age 55–70', df_results[(df_results.age>=55)&(df_results.age<70)]),
    ('Age 70+', df_results[df_results.age>=70]),
    ('Black/African', df_results[df_results.black_african==1]),
    ('Hispanic', df_results[df_results.hispanic==1]),
    ('Other/White', df_results[(df_results.black_african==0)&(df_results.hispanic==0)]),
    ('Diabetic (HbA1c≥6.5)', df_results[df_results.hba1c>=6.5] if 'hba1c' in df_results else df_results[:0]),
    ('Non-diabetic', df_results[df_results.hba1c<6.5] if 'hba1c' in df_results else df_results),
    ('On BP medication', df_results[df_results.on_bp_medication==1]),
    ('No BP medication', df_results[df_results.on_bp_medication==0]),
]

print('SUBGROUP ANALYSIS — MIMIC-IV External Validation')
print('='*76)
print(f' {"Subgroup":30s} {"N":>6} {"CVD%":>5} {"Fusion AUC":>10} {"ECG AUC":>8} {"Recall":>7}')
print(' '+'─'*72)

subgroup_results = []
for name, sub in subgroups:
    if len(sub) < 30 or sub['label'].sum() < 5:
        continue
    try:
        f_auc = roc_auc_score(sub['label'], sub['fusion_pred'])
        e_auc = roc_auc_score(sub['label'], sub['ecg_pred'])
        rec = recall_score(sub['label'], (sub['fusion_pred']>=THRESH).astype(int))
        n = len(sub)
        cvd = sub['label'].mean()*100
        print(f' {name:30s} {n:6,} {cvd:5.1f}% {f_auc:10.4f} {e_auc:8.4f} {rec:7.3f}')
        subgroup_results.append({'subgroup':name,'n':n,'cvd_pct':cvd,
                                  'fusion_auc':f_auc,'ecg_auc':e_auc,'recall':rec})
    except Exception as e:
        print(f' {name:30s} {e}')
print('='*76)
print()
print(' Flag if any subgroup AUC differs by >0.05 from overall — equity concern')
df_sub = pd.DataFrame(subgroup_results)
df_sub.to_csv(f'{MIMIC_DIR}/subgroup_results.csv', index=False)
print(f'Subgroup results saved to Drive.')

## 8. Full External Validation Dashboard

In [ ]:
def style(ax):
    ax.set_facecolor('#111D2E')
    for s in ax.spines.values(): s.set_edgecolor('#1B3A6B')
    ax.tick_params(colors='#8892A4', labelsize=8)
    ax.grid(True, alpha=0.15, color='#8892A4', linewidth=0.5)
    return ax

fig = plt.figure(figsize=(20, 18), facecolor='#0A1628')
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.50, wspace=0.38)

# ── 1. ROC Curve — all three branches ─────────────────────────────────────
ax = style(fig.add_subplot(gs[0, 0]))
for preds_r, lbl_r, col_r in [
    (fusion_preds, f'Fusion AUC={fusion_auc:.4f}', '#E63946'),
    (ecg_preds, f'ECG AUC={ecg_auc:.4f}', '#00B4D8'),
    (clin_preds, f'Clinical AUC={clin_auc:.4f}', '#06D6A0'),
]:
    if np.isnan(preds_r).any(): continue
    fpr_r,tpr_r,_ = roc_curve(labels, preds_r)
    ax.plot(fpr_r, tpr_r, linewidth=2, label=lbl_r)
ax.plot([0,1],[0,1],'--',color='#4A5568')
ax.set_title('ROC Curve — MIMIC-IV External\n(Real ECG + Real Clinical)',
             color='white', fontsize=10)
ax.set_xlabel('FPR', color='#8892A4'); ax.set_ylabel('TPR', color='#8892A4')
ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=7)

# ── 2. PR Curve ────────────────────────────────────────────────────────────
ax = style(fig.add_subplot(gs[0, 1]))
ax.step(rec_f, prec_f, color='#E63946', linewidth=2, where='post',
        label=f'Fusion AP={fusion_ap:.4f}')
ax.fill_between(rec_f, prec_f, alpha=0.12, color='#E63946', step='post')
ax.axhline(labels.mean(), color='#4A5568', linewidth=1.5,
           linestyle='--', label=f'Baseline ({labels.mean():.3f})')
ax.set_title('Precision-Recall Curve', color='white', fontsize=11)
ax.set_xlabel('Recall', color='#8892A4'); ax.set_ylabel('Precision', color='#8892A4')
ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=8)

# ── 3. Confusion Matrix ────────────────────────────────────────────────────
ax = style(fig.add_subplot(gs[0, 2]))
cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True)
sns.heatmap(cm_n, annot=True, fmt='.2f', ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=['Pred: No CVD','Pred: CVD'],
            yticklabels=['True: No CVD','True: CVD'], linewidths=0.5)
ax.set_title(f'Confusion Matrix (t={THRESH:.3f})\nTP={tp} FP={fp} FN={fn} TN={tn}',
             color='white', fontsize=10)
ax.tick_params(colors='white', labelsize=8)

# ── 4. Calibration ─────────────────────────────────────────────────────────
ax = style(fig.add_subplot(gs[0, 3]))
for preds_c, lbl_c, col_c in [
    (fusion_preds,'Fusion', '#E63946'),
    (clin_preds, 'Clinical', '#06D6A0'),
]:
    if len(preds_c) > 0:
        fp2, mp2 = calibration_curve(labels, preds_c, n_bins=10)
        ax.plot(mp2, fp2, 's-', color=col_c, linewidth=2, label=lbl_c, markersize=5)
ax.plot([0,1],[0,1],'k--',color='#8892A4', linewidth=1, label='Perfect')
ax.set_title('Probability Calibration\n(MIMIC-IV real data)', color='white', fontsize=11)
ax.set_xlabel('Predicted probability', color='#8892A4')
ax.set_ylabel('Fraction positives', color='#8892A4')
ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=8)

# ── 5. Score distributions ─────────────────────────────────────────────────
ax = style(fig.add_subplot(gs[1, 0:2]))
ax.hist(fusion_preds[labels==0], bins=40, color='#00B4D8', alpha=0.7,
        label=f'No CVD (n={(labels==0).sum():,})', density=True)
ax.hist(fusion_preds[labels==1], bins=40, color='#E63946', alpha=0.7,
        label=f'CVD (n={(labels==1).sum():,})', density=True)
ax.axvline(THRESH, color='#FFB703', linewidth=2.5, linestyle='--',
           label=f'Threshold {THRESH:.3f}')
ax.set_title('Fusion Score Distribution — MIMIC-IV', color='white', fontsize=11)
ax.set_xlabel('Predicted CVD probability', color='#8892A4')
ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=8)

# ── 6. Attention gate on real data ─────────────────────────────────────────
ax = style(fig.add_subplot(gs[1, 2]))
ax.hist(alphas, bins=30, color='#E63946', alpha=0.7, label=f'α ECG μ={alphas.mean():.3f}', density=True)
ax.hist(betas, bins=30, color='#00B4D8', alpha=0.7, label=f'β Clin μ={betas.mean():.3f}', density=True)
ax.axvline(alphas.mean(), color='#E63946', linewidth=2.5, linestyle='--')
ax.axvline(betas.mean(), color='#00B4D8', linewidth=2.5, linestyle='--')
ax.set_title('Attention Gate — Real MIMIC-IV Data\n(did α adapt to real clinical quality?)',
             color='white', fontsize=10)
ax.set_xlabel('Gate weight', color='#8892A4')
ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=8)

# ── 7. Subgroup AUC bar chart ──────────────────────────────────────────────
ax = style(fig.add_subplot(gs[1, 3]))
if len(df_sub) > 0:
    sub_names = [s[:15] for s in df_sub['subgroup'].values]
    sub_aucs = df_sub['fusion_auc'].values
    colors_sg = ['#E63946' if v < fusion_auc - 0.05 else '#06D6A0' for v in sub_aucs]
    bars = ax.barh(sub_names, sub_aucs, color=colors_sg, alpha=0.85, edgecolor='#1B3A6B')
    ax.axvline(fusion_auc, color='#FFB703', linewidth=2, linestyle='--',
               label=f'Overall {fusion_auc:.4f}')
    ax.axvline(fusion_auc-0.05, color='#E63946', linewidth=1.5, linestyle=':',
               alpha=0.7, label='Equity threshold (-0.05)')
    ax.set_title('Subgroup AUC\n(Red = equity concern)',
                 color='white', fontsize=10)
    ax.set_xlabel('AUC', color='#8892A4')
    ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=7)

# ── 8. Full model comparison including external validation ─────────────────
ax = style(fig.add_subplot(gs[2, :]))
comparison = [
    ('RF+SVM\nFramingham\n(your NB)', 0.72, 'Train+Test', '#4A5568'),
    ('XGBoost\nNHANES NB03b', clin_cfg['test_auc'], 'Internal', '#00B4D8'),
    ('ECG NB02\nPTB-XL', ecg_ckpt['mi_auc'], 'Internal', '#00B4D8'),
    ('Fusion NB04\nPTB-XL synthetic', fuse_ckpt['val_auc'], 'Internal', '#FFB703'),
    ('Fusion NB02d\nMIMIC-IV REAL', fusion_auc, 'External ', '#E63946'),
]
x_pos = np.arange(len(comparison))
aucs_cmp = [m[1] for m in comparison]
cols_cmp = [m[3] for m in comparison]
bars = ax.bar(x_pos, aucs_cmp, color=cols_cmp, alpha=0.85, edgecolor='#1B3A6B')
ax.axhline(0.85, color='#E63946', linewidth=2, linestyle='--', label='Target AUC 0.85')
for bar,v in zip(bars, aucs_cmp):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.4f}',
            ha='center', color='white', fontsize=10, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([m[0] for m in comparison], color='white', fontsize=9)
ax.set_ylim(0.5, 1.05)
ax.set_title('Complete Model Validation — Internal vs External',
             color='white', fontsize=12)
ax.set_ylabel('AUC-ROC', color='#8892A4')
ax.legend(facecolor='#0A1628', labelcolor='white', fontsize=9)
# Annotate internal vs external
for i,m in enumerate(comparison):
    ax.text(i, 0.52, m[2], ha='center', color='#8892A4', fontsize=8)

plt.suptitle('KardioSense Fusion Model — MIMIC-IV External Validation\n'
             f'n={len(labels):,} real ECG+Clinical records '
             f'(Beth Israel Deaconess, Boston)',
             color='white', fontsize=13, y=1.01)
plt.savefig(f'{LOG_DIR}/nb02d_mimic_validation.png', dpi=120,
            bbox_inches='tight', facecolor='#0A1628')
plt.show()
print('Dashboard saved.')

## 9. Save Results

In [ ]:
# Final save
np.savez_compressed(RESULTS_PATH,
    ecg_preds = ecg_preds,
    clin_preds = clin_preds,
    fusion_preds = fusion_preds,
    labels = labels,
    mi_labels = mi_labels,
    afib_labels = afib_labels,
    alphas = alphas,
    betas = betas,
    n_processed = np.array(len(labels)))

validation_summary = {
    'dataset' : 'MIMIC-IV ECG (Beth Israel Deaconess, Boston)',
    'n_records' : int(len(labels)),
    'cvd_event_rate' : float(labels.mean()),
    'fusion_auc' : float(fusion_auc),
    'ecg_auc' : float(ecg_auc),
    'clinical_auc' : float(clin_auc),
    'fusion_ap' : float(fusion_ap),
    'fusion_brier' : float(fusion_brier),
    'recall_clinical_threshold': float(fusion_rec),
    'threshold' : float(THRESH),
    'TP':int(tp), 'FP':int(fp), 'FN':int(fn), 'TN':int(tn),
    'alpha_mean' : float(alphas.mean()),
    'beta_mean' : float(betas.mean()),
    'n_errors' : n_errors,
    'n_flat_signals' : n_flat,
    'comparison' : {
        'ptbxl_mi_auc' : float(ecg_ckpt['mi_auc']),
        'nhanes_clin_auc' : float(clin_cfg['test_auc']),
        'fusion_train_auc': float(fuse_ckpt['val_auc']),
        'mimic_ext_auc' : float(fusion_auc),
    }
}
with open(f'{MIMIC_DIR}/mimic_validation_summary.json', 'w') as f:
    json.dump(validation_summary, f, indent=2)

print(' Results saved:')
print(f' {MIMIC_DIR}/mimic_validation_results.npz')
print(f' {MIMIC_DIR}/mimic_validation_summary.json')
print(f' {MIMIC_DIR}/subgroup_results.csv')
print(f' {LOG_DIR}/nb02d_mimic_validation.png')
print()
print(' FINAL EXTERNAL VALIDATION SUMMARY:')
print(f' Dataset : MIMIC-IV ECG — Beth Israel Deaconess, Boston')
print(f' N validated : {len(labels):,} real paired ECG+clinical records')
print(f' Fusion AUC : {fusion_auc:.4f}')
print(f' ECG AUC : {ecg_auc:.4f}')
print(f' Clinical AUC: {clin_auc:.4f}')
print(f' Recall : {fusion_rec:.3f} (at threshold {THRESH:.3f})')
print()
print('This is your publishable external validation result.')
print('Three independent datasets now validate KardioSense:')
print(f' 1. CODE-15% (Brazil) — ECG external validation (NB02b)')
print(f' 2. NHANES (USA) — Clinical branch validation (NB03b)')
print(f' 3. MIMIC-IV (USA, real) — Full fusion external validation (this)')
print()
print(' Next: Notebook 05 — TFLite Export for device deployment')